In [0]:
ruta = "/Volumes/workspace/datalake/landing/sap/empresas/empresa.data"

df = (
    spark.read
    .option("header", "true")
    .option("delimiter", "|")
    .schema("ID STRING, NOMBRE STRING")
    .csv(ruta)
)

display(df)
print("Total de registros:", df.count())

In [0]:
tabla = "workspace.bronze.empresa"

registros_actuales = spark.table(tabla).count()

if registros_actuales == 0:
    df.write.format("delta").mode("append").saveAsTable(tabla)
    print("Datos guardados correctamente.")
else:
    print("La tabla ya tiene datos. Se omite la carga para evitar duplicados.")

display(spark.table(tabla))
print("Total en Bronze:", spark.table(tabla).count())

In [0]:
ruta_personas = "/Volumes/workspace/datalake/landing/sap/personas/persona.data"

df_personas = (
    spark.read
    .option("header", "true")
    .option("delimiter", "|")
    .schema("""
        ID STRING,
        NOMBRE STRING,
        TELEFONO STRING,
        CORREO STRING,
        FECHA_INGRESO STRING,
        EDAD STRING,
        SALARIO STRING,
        ID_EMPRESA STRING
    """)
    .csv(ruta_personas)
)

display(df_personas)

print("Total de personas:", df_personas.count())

In [0]:
tabla_personas = "workspace.bronze.persona"

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.persona (
    ID STRING,
    NOMBRE STRING,
    TELEFONO STRING,
    CORREO STRING,
    FECHA_INGRESO STRING,
    EDAD STRING,
    SALARIO STRING,
    ID_EMPRESA STRING
)
USING DELTA
""")

registros_actuales = spark.table(tabla_personas).count()

if registros_actuales == 0:
    df_personas.write.format("delta").mode("append").saveAsTable(tabla_personas)
    print("Personas guardadas correctamente.")
else:
    print("La tabla ya contiene datos. No se insertaron duplicados.")

display(spark.table(tabla_personas))

print("Total de personas en Bronze:", spark.table(tabla_personas).count())

In [0]:
ruta_transacciones = "/Volumes/workspace/datalake/landing/sap/transacciones/transacciones.data"

df_transacciones = (
    spark.read
    .option("header", "true")
    .option("delimiter", "|")
    .schema("""
        ID_PERSONA STRING,
        ID_EMPRESA STRING,
        MONTO STRING,
        FECHA STRING
    """)
    .csv(ruta_transacciones)
)

display(df_transacciones.limit(20))

print("Total de transacciones:", df_transacciones.count())

In [0]:

tabla_transacciones = "workspace.bronze.transacciones"

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.transacciones (
    ID_PERSONA STRING,
    ID_EMPRESA STRING,
    MONTO STRING,
    FECHA STRING
)
USING DELTA
""")

registros_actuales = spark.table(tabla_transacciones).count()

if registros_actuales == 0:
    (
        df_transacciones.write
        .format("delta")
        .mode("append")
        .saveAsTable(tabla_transacciones)
    )
    print("Transacciones guardadas correctamente.")
else:
    print("La tabla ya tiene datos. Se omite la carga.")

display(spark.table(tabla_transacciones).limit(20))

print(
    "Total en Bronze:",
    spark.table(tabla_transacciones).count()
)

In [0]:
resultado = spark.sql("""
    SELECT 'empresa' AS tabla, COUNT(*) AS registros
    FROM workspace.bronze.empresa

    UNION ALL

    SELECT 'persona' AS tabla, COUNT(*) AS registros
    FROM workspace.bronze.persona

    UNION ALL

    SELECT 'transacciones' AS tabla, COUNT(*) AS registros
    FROM workspace.bronze.transacciones
""")

display(resultado)

In [0]:
from pyspark.sql import functions as F

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.silver
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.empresa (
    ID INT,
    NOMBRE STRING,
    FECHA_PROCESO DATE,
    PERIODO STRING
)
USING DELTA
PARTITIONED BY (PERIODO)
""")

df_silver_empresa = (
    spark.table("workspace.bronze.empresa")
    .select(
        F.col("ID").cast("int").alias("ID"),
        F.trim(F.col("NOMBRE")).alias("NOMBRE")
    )
    .withColumn("FECHA_PROCESO", F.current_date())
    .withColumn(
        "PERIODO",
        F.date_format(F.current_date(), "yyyyMM")
    )
)

destino = "workspace.silver.empresa"

if spark.table(destino).count() == 0:
    (
        df_silver_empresa.write
        .format("delta")
        .mode("append")
        .saveAsTable(destino)
    )
    print("Empresas guardadas correctamente en Silver.")
else:
    print("La tabla ya contiene registros. No se duplicaron.")

display(spark.table(destino))

print("Total Silver:", spark.table(destino).count())


In [0]:
from pyspark.sql import functions as F

destino = "workspace.silver.persona"

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.persona (
    ID INT,
    NOMBRE STRING,
    TELEFONO STRING,
    CORREO STRING,
    FECHA_INGRESO DATE,
    EDAD INT,
    SALARIO DOUBLE,
    ID_EMPRESA INT,
    EDAD_CATEGORIA STRING,
    FECHA_PROCESO DATE,
    PERIODO STRING,
    ANIO INT,
    MES INT,
    DIA INT,
    DIA_TEXTO STRING
)
USING DELTA
PARTITIONED BY (PERIODO)
""")

df_bronze_persona = spark.table(
    "workspace.bronze.persona"
)

df_silver_persona = (
    df_bronze_persona
    .select(
        F.col("ID").cast("int").alias("ID"),
        F.trim("NOMBRE").alias("NOMBRE"),
        F.regexp_replace("TELEFONO", "-", "").alias("TELEFONO"),
        F.col("CORREO"),
        F.to_date("FECHA_INGRESO", "yyyy-MM-dd").alias("FECHA_INGRESO"),
        F.col("EDAD").cast("int").alias("EDAD"),
        F.col("SALARIO").cast("double").alias("SALARIO"),
        F.col("ID_EMPRESA").cast("int").alias("ID_EMPRESA")
    )
    .withColumn(
        "EDAD_CATEGORIA",
        F.when(F.col("EDAD") < 30, "Joven")
         .when(F.col("EDAD") <= 50, "Adulto")
         .otherwise("Adulto mayor")
    )
    .withColumn("FECHA_PROCESO", F.current_date())
    .withColumn("PERIODO", F.date_format(F.current_date(), "yyyyMM"))
    .withColumn("ANIO", F.year(F.current_date()))
    .withColumn("MES", F.month(F.current_date()))
    .withColumn("DIA", F.dayofmonth(F.current_date()))
    .withColumn("DIA_TEXTO", F.date_format(F.current_date(), "EEEE"))
)

if spark.table(destino).count() == 0:
    (
        df_silver_persona.write
        .format("delta")
        .mode("append")
        .saveAsTable(destino)
    )
    print("Personas guardadas correctamente en Silver.")
else:
    print("La tabla ya contiene datos. No se realizo otra carga.")

display(spark.table(destino))

print("Total Silver personas:", spark.table(destino).count())

In [0]:

from pyspark.sql import functions as F

destino_trx = "workspace.silver.transacciones"

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.transacciones (
    ID_PERSONA INT,
    ID_EMPRESA INT,
    MONTO DOUBLE,
    FECHA DATE,
    ANIO INT,
    MES INT,
    DIA INT,
    PERIODO STRING
)
USING DELTA
PARTITIONED BY (PERIODO)
""")

df_bronze_trx = spark.table(
    "workspace.bronze.transacciones"
)

df_silver_trx = (
    df_bronze_trx
    .select(
        F.col("ID_PERSONA").cast("int").alias("ID_PERSONA"),
        F.col("ID_EMPRESA").cast("int").alias("ID_EMPRESA"),
        F.col("MONTO").cast("double").alias("MONTO"),
        F.to_date("FECHA", "yyyy-MM-dd").alias("FECHA")
    )
    .withColumn("ANIO", F.year("FECHA"))
    .withColumn("MES", F.month("FECHA"))
    .withColumn("DIA", F.dayofmonth("FECHA"))
    .withColumn(
        "PERIODO",
        F.date_format(F.current_date(), "yyyyMM")
    )
)

columnas = ["ID_PERSONA", "ID_EMPRESA", "MONTO", "FECHA"]

invalidos = df_silver_trx.filter(
    F.col("ID_PERSONA").isNull()
    | F.col("ID_EMPRESA").isNull()
    | F.col("MONTO").isNull()
    | F.col("FECHA").isNull()
).count()

print("Registros con valores nulos:", invalidos)

if invalidos > 0:
    raise ValueError("Revisar datos antes de cargar Silver.")

if spark.table(destino_trx).count() == 0:
    (
        df_silver_trx.write
        .format("delta")
        .mode("append")
        .saveAsTable(destino_trx)
    )
    print("Transacciones guardadas correctamente en Silver.")
else:
    print("La tabla ya contiene datos. No se repite la carga.")

display(spark.table(destino_trx).limit(20))

print(
    "Total Silver transacciones:",
    spark.table(destino_trx).count()
)

In [0]:

from pyspark.sql import functions as F

df_gold = spark.sql("""
SELECT
    p.PERIODO,
    e.NOMBRE AS EMPRESA,
    p.EDAD_CATEGORIA,
    AVG(p.EDAD) AS PROM_EDAD,
    AVG(p.SALARIO) AS PROM_SALARIO,
    SUM(p.SALARIO) AS PLANILLA,
    COUNT(*) AS NUM_EMPLEADOS,
    MAX(p.SALARIO) AS MAX_SALARIO,
    MIN(p.SALARIO) AS MIN_SALARIO
FROM workspace.silver.persona p
INNER JOIN workspace.silver.empresa e
    ON p.ID_EMPRESA = e.ID
    AND p.PERIODO = e.PERIODO
GROUP BY
    p.PERIODO,
    e.NOMBRE,
    p.EDAD_CATEGORIA
""")

display(df_gold.orderBy("EMPRESA", "EDAD_CATEGORIA"))

total_incluidos = df_gold.agg(
    F.sum("NUM_EMPLEADOS")
).first()[0]

print("Empleados incluidos en Gold:", total_incluidos)
print(
    "Empleados en Silver:",
    spark.table("workspace.silver.persona").count()
)

In [0]:

from pyspark.sql import functions as F

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.gold
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.kpi_empleados (
    PERIODO STRING,
    EMPRESA STRING,
    EDAD_CATEGORIA STRING,
    PROM_EDAD DOUBLE,
    PROM_SALARIO DOUBLE,
    PLANILLA DOUBLE,
    NUM_EMPLEADOS BIGINT,
    MAX_SALARIO DOUBLE,
    MIN_SALARIO DOUBLE
)
USING DELTA
PARTITIONED BY (PERIODO)
""")

destino_gold = "workspace.gold.kpi_empleados"

if spark.table(destino_gold).count() == 0:
    (
        df_gold.write
        .format("delta")
        .mode("append")
        .saveAsTable(destino_gold)
    )
    print("Indicadores guardados en Gold.")
else:
    print("La tabla ya contiene datos. No se repitió la carga.")

tabla_gold = spark.table(destino_gold)

display(tabla_gold.orderBy("EMPRESA", "EDAD_CATEGORIA"))

print("Total de grupos KPI:", tabla_gold.count())

total_empleados = tabla_gold.agg(
    F.sum("NUM_EMPLEADOS")
).first()[0]

print("Empleados incluidos:", total_empleados)


In [0]:
tablas = [
    ("Bronze", "empresa", 10),
    ("Bronze", "persona", 100),
    ("Bronze", "transacciones", 235040),
    ("Silver", "empresa", 10),
    ("Silver", "persona", 100),
    ("Silver", "transacciones", 235040),
    ("Gold", "kpi_empleados", 30)
]

resultados = []

for capa, nombre, esperado in tablas:
    ruta = f"workspace.{capa.lower()}.{nombre}"
    cantidad = spark.table(ruta).count()

    estado = (
        "CORRECTO" if cantidad == esperado
        else "REVISAR"
    )

    resultados.append(
        (capa, nombre, cantidad, esperado, estado)
    )

df_auditoria = spark.createDataFrame(
    resultados,
    ["CAPA", "TABLA", "REGISTROS",
     "ESPERADOS", "ESTADO"]
)

display(df_auditoria)

from pyspark.sql import functions as F

empleados_gold = (
    spark.table("workspace.gold.kpi_empleados")
    .agg(F.sum("NUM_EMPLEADOS"))
    .first()[0]
)

print("Empleados incluidos en Gold:", empleados_gold)

if all(r[4] == "CORRECTO" for r in resultados) and empleados_gold == 100:
    print("VALIDACION DE CANTIDADES COMPLETADA")
else:
    print("REVISAR RESULTADOS")